### Scott 10k parcellate

#### Aim: take the z-scored, 1mm3 MNI152 mwc1t1 scans and parcellating them with either:

1. Desikan Killiany Atlas + aseg
2. Destrieux Atlas + aseg

#### To retrieve ONLY grey matter values for each scan

#### (Optional) Prelims: remove destrieux + desikan patientwise parcellations if needed

In [ ]:
%%bash

destrieux_file=/rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_lists/successful_destrieux_parcellations.txt
desikan_file=/rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_lists/successful_desikan_parcellations.txt

rm $(cat $destrieux_file)
rm $(cat $desikan_file)


#### Prelims: make both atlases
#### Make scripts and list folders

In [1]:
%%bash
mkdir -p /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate

#### 1. Recon-all an MNI152 brain

In [3]:
%%file /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/reconall_mni152_1mm.sh

#!/bin/bash
#PBS -l walltime=72:00:00
#PBS -l select=1:ncpus=1:mem=30gb
#PBS -N reconall_MNI_1mm
#PBS -o /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_logs/recon_all/
#PBS -e /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_logs/recon_all/

inpath="/rds/general/project/c3nl_scott_students/live/sankeith/standards/MNI152_T1_1mm_brain.nii.gz"
infile=`basename $inpath .nii.gz`;


outdir=/rds/general/project/c3nl_scott_students/ephemeral/sankeith/reconall_mni152_1mm
mkdir -p "$outdir"

module load tools/prod
module load FreeSurfer/7.4.1-centos8_x86_64 > /dev/null 2>&1
module load FSL/6.0.5.1-foss-2021a > /dev/null 2>&1
module load MATLAB/2024b > /dev/null 2>&1

export JOB_NUM=$(echo ${PBS_JOBID} | cut -f 1 -d '.' | cut -f 1 -d '[')
export NEW_TMPDIR="${EPHEMERAL}/${JOB_NUM}.${PBS_ARRAY_INDEX}"
mkdir -p ${NEW_TMPDIR}
export TMPDIR=${NEW_TMPDIR}
export SUBJECTS_DIR="/rds/general/project/c3nl_scott_students/live/sankeith/standards/"
export FS_LICENSE="${HOME}/license.txt"

cmd="recon-all -all -i ${inpath} -subjid ${infile} -sd ${outdir}"
echo ${cmd}
${cmd}

Writing /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/reconall_mni152_1mm.sh


In [43]:
%%bash

chmod -Rf 775 /rds/general/project/c3nl_scott_students/live/sankeith/scott_10k_housekeeping/reconall_mni152_1mm.sh
qsub /rds/general/project/c3nl_scott_students/live/sankeith/scott_10k_housekeeping/reconall_mni152_1mm.sh

1800746.pbs-7


#### Take the desikan-kiliany and destrieux mgz files and:

1. Turn them to .nii.gz files
2. Make sure they are in MNI152 orientation
3. FLIRT them to the MNI152 images and in 1mm^3 resolution

In [1]:
%%bash

module purge
module load tools/prod
module load FSL/6.0.7.17 > /dev/null 2>&1
module load FreeSurfer/7.4.1-centos8_x86_64 > /dev/null 2>&1

source $FSLDIR/etc/fslconf/fsl.sh
export PATH=$FSLDIR/bin:$PATH
export FSLOUTPUTTYPE="NIFTI_GZ"

arr=("aparc+aseg" "aparc.a2009s+aseg")
final_paths=("MNI152_1mm_desikan_aparc+aseg" "MNI152_1mm_destrieux_aparc.a2009+aseg")

ref="/rds/general/project/c3nl_scott_students/live/sankeith/standards/MNI152_T1_1mm_brain.nii.gz"
base_in="/rds/general/project/c3nl_scott_students/ephemeral/sankeith/reconall_mni152_1mm/MNI152_T1_1mm_brain/mri"
base_out="/rds/general/project/c3nl_scott_students/live/sankeith/standards"

for idx in "${!arr[@]}"
do
    label="${arr[$idx]}"
    final="${final_paths[$idx]}"

    inpath="${base_in}/${label}.mgz"
    out1="${base_out}/${label}.mgz"
    out2="${base_out}/${label}.nii.gz"
    out3="${base_out}/mni_${label}.nii.gz"
    final_out="${base_out}/${final}.nii.gz"

    echo "Processing $label"

    rsync -av "$inpath" "$out1"

    mri_convert "$out1" "$out2"

    fslreorient2std "$out2" "$out3"

    flirt -in "$out3" \
          -ref "$ref" \
          -applyxfm \
          -usesqform \
          -interp nearestneighbour \
          -out "$final_out"

    fslinfo "$final_out";


    echo "Created: $final_out";
done

Processing aparc+aseg
sending incremental file list


rsync: change_dir "/rds/general/project/c3nl_scott_students/ephemeral/sankeith/reconall_mni152_1mm/MNI152_T1_1mm_brain/mri" failed: No such file or directory (2)



sent 20 bytes  received 12 bytes  64.00 bytes/sec
total size is 0  speedup is 0.00


rsync error: some files/attrs were not transferred (see previous errors) (code 23) at main.c(1187) [sender=3.1.3]


mri_convert /rds/general/project/c3nl_scott_students/live/sankeith/standards/aparc+aseg.mgz /rds/general/project/c3nl_scott_students/live/sankeith/standards/aparc+aseg.nii.gz 
reading from /rds/general/project/c3nl_scott_students/live/sankeith/standards/aparc+aseg.mgz...
000.00, TE=0.00, TI=0.00, flip angle=0.00
i_ras = (-1, 0, 0)
j_ras = (0, 0, -1)
k_ras = (0, 1, 0)
iting to /rds/general/project/c3nl_scott_students/live/sankeith/standards/aparc+aseg.nii.gz...
data_type	INT32
dim1		182
dim2		218
dim3		182
dim4		1
datatype	8
pixdim1		1.000000
pixdim2		1.000000
pixdim3		1.000000
pixdim4		1.000000
cal_max		0.000000
cal_min		0.000000
file_type	NIFTI-1+
Created: /rds/general/project/c3nl_scott_students/live/sankeith/standards/MNI152_1mm_desikan_aparc+aseg.nii.gz
Processing aparc.a2009s+aseg
sending incremental file list


rsync: change_dir "/rds/general/project/c3nl_scott_students/ephemeral/sankeith/reconall_mni152_1mm/MNI152_T1_1mm_brain/mri" failed: No such file or directory (2)



sent 20 bytes  received 12 bytes  64.00 bytes/sec
total size is 0  speedup is 0.00


rsync error: some files/attrs were not transferred (see previous errors) (code 23) at main.c(1187) [sender=3.1.3]


mri_convert /rds/general/project/c3nl_scott_students/live/sankeith/standards/aparc.a2009s+aseg.mgz /rds/general/project/c3nl_scott_students/live/sankeith/standards/aparc.a2009s+aseg.nii.gz 
reading from /rds/general/project/c3nl_scott_students/live/sankeith/standards/aparc.a2009s+aseg.mgz...
TR=1000.00, TE=0.00, TI=0.00, flip angle=0.00
i_ras = (-1, 0, 0)
j_ras = (0, 0, -1)
, 0)s = (0, 1
ii.gz...to /rds/general/project/c3nl_scott_students/live/sankeith/standards/aparc.a2009s+aseg.n
data_type	INT32
dim1		182
dim2		218
dim3		182
dim4		1
datatype	8
pixdim1		1.000000
pixdim2		1.000000
pixdim3		1.000000
pixdim4		1.000000
cal_max		0.000000
cal_min		0.000000
file_type	NIFTI-1+
Created: /rds/general/project/c3nl_scott_students/live/sankeith/standards/MNI152_1mm_destrieux_aparc.a2009+aseg.nii.gz


#### Gather z-scored, FLIRTed, mwc1t1 scan paths

In [2]:
%%bash

mydir=/rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_20aug25/
output_file=/rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_lists/successful_zscored_mwc1t1s.txt

### collect z-scored FLIRTed file paths##

> "$output_file"
find "$mydir" -type f -name "Z_F*nii*" >> "$output_file"
wc -l < "$output_file"

15725


#### Extract atlas RoIs

In [10]:
%%bash

module load fsl > /dev/null 2>&1
module load FSL/6.0.5.1-foss-2021a > /dev/null 2>&1

outdir=/rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_parcellate
des_atlas=/rds/general/project/c3nl_scott_students/live/sankeith/standards/MNI152_1mm_destrieux_aparc.a2009+aseg.nii.gz
dk_atlas=/rds/general/project/c3nl_scott_students/live/sankeith/standards/MNI152_1mm_desikan_aparc+aseg.nii.gz

mkdir -p "$outdir"/atlas_roi_extraction

fslmeants -i "$des_atlas" --label="$des_atlas" -o "$outdir"/'atlas_roi_extraction'/'destrieux+aseg_extraction.txt'
fslmeants -i "$dk_atlas" --label="$dk_atlas" -o "$outdir"/'atlas_roi_extraction'/'desikan+aseg_extraction.txt'


#### Turn a lookup table into a dictionary for use later

In [13]:
from itertools import islice
import pandas as pd
import os
import math
import pickle

root = '/rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_parcellate/atlas_roi_extraction/'
lut_df = pd.read_csv('/rds/general/project/c3nl_scott_students/live/sankeith/standards/FreeSurferColorLUTLabels.csv', low_memory = False, header = None)

##Turn the FreeSurfer LUT table into a dictionary
lut_df.rename(columns = {0:'ID', 1:'ROI'}, inplace = True)
lut_df['ID'] = lut_df['ID'].round(0)
print(lut_df.shape)
print(lut_df['ID'].dtype)

lut_dict_keys = lut_df['ID'].tolist()
lut_dict_vals = lut_df['ROI'].tolist()

lut_dict = dict(zip(lut_dict_keys, lut_dict_vals))
lut_dict['DATA_KEY'] = 'DATA_KEY'

##Check that column headings aren't in the dictionary, then pickle it
lut_dict_slice = dict(islice(lut_dict.items(), 2))
print(lut_dict_slice)

pickle.dump(lut_dict, open("/rds/general/project/c3nl_scott_students/live/sankeith/standards/lut_dict.pkl", 'wb'))

(1201, 2)
float64
{0.0: 'Unknown', 1.0: 'Left-Cerebral-Exterior'}


#### (Pickle the extracted atlas ROIs as a list for use later)

In [14]:
import pandas as pd
import os, csv, pickle

root = '/rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_parcellate/atlas_roi_extraction/'


for txt in ['destrieux+aseg_extraction', 'desikan+aseg_extraction']:
    path = os.path.join(root, txt + '.txt')
    csv_path = os.path.join(root, txt + '.csv')
    if os.path.exists(path):
        df = pd.read_csv(path, sep=r'\s+', header= None)
        print(df.iloc[0,:10])
        df.to_csv(csv_path, index = None, columns = None)
        with open(csv_path, 'r') as f:
            data = list(csv.reader(f, delimiter = ','))
            data = data[1]
            print(f"Data to be pickled is of type {type(data)}"
                  f"\n{data[:20]}")
            f.close()
        pickle_path = os.path.join(root, f"{txt}.pkl")
        pickle.dump(data, open(pickle_path, 'wb'))

0     0
1     2
2     0
3     4
4     5
5     0
6     7
7     8
8     0
9    10
Name: 0, dtype: int64
Data to be pickled is of type <class 'list'>
['0', '2', '0', '4', '5', '0', '7', '8', '0', '10', '11', '12', '13', '14', '15', '16', '17', '18', '0', '0']
0     0
1     2
2     0
3     4
4     5
5     0
6     7
7     8
8     0
9    10
Name: 0, dtype: int64
Data to be pickled is of type <class 'list'>
['0', '2', '0', '4', '5', '0', '7', '8', '0', '10', '11', '12', '13', '14', '15', '16', '17', '18', '0', '0']


#### Extract ROIs from z-scored images

In [ ]:
%%file /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/scott_10k_parcellate.sh

#!/bin/bash
#PBS -l walltime=00:10:00
#PBS -l select=1:ncpus=1:mem=20gb
#PBS -N scott_10k_parcellate
#PBS -J 0-9999
#PBS -o /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_logs/b2c/
#PBS -e /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_logs/b2c/

module load tools/prod
module load fsl > /dev/null 2>&1
module load FSL/6.0.7.17 > /dev/null 2>&1
source ${FSLDIR}/etc/fslconf/fsl.sh
export FSLOUTPUTTYPE=NIFTI_GZ

inpath=`head -n ${PBS_ARRAY_INDEX} /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_lists/successful_zscored_mwc1t1s.txt | tail -n 1`
des_atlas=/rds/general/project/c3nl_scott_students/live/sankeith/standards/MNI152_1mm_destrieux_aparc.a2009+aseg.nii.gz
dk_atlas=/rds/general/project/c3nl_scott_students/live/sankeith/standards/MNI152_1mm_desikan_aparc+aseg.nii.gz
outdir=/rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_20aug25
folderbase=$(basename "$(dirname "$inpath")")
outroot="$outdir"/"$folderbase"
echo "inpath: $inpath";
echo "folderbase: $folderbase";
echo "outroot: $outroot"

cmd="fslmeants -i $inpath --label=$des_atlas -o ${outroot}/destrieux_mwc1t1_${folderbase}.txt"
echo ${cmd}
eval ${cmd}

cmd2="fslmeants -i $inpath --label=$dk_atlas -o ${outroot}/desikan_mwc1t1_${folderbase}.txt"
echo ${cmd2}
eval ${cmd2}

In [5]:
%%bash

chmod -Rf 775 /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/scott_10k_parcellate.sh
export PBS_ARRAY_INDEX=1; /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/scott_10k_parcellate.sh
qsub /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/scott_10k_parcellate.sh

inpath: /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_20aug25/128_S_0216_ADNI-T1_2006-09-12_07_31_03.0/Z_F_1mm_128_S_0216_ADNI-T1_2006-09-12_07_31_03.0_dicv_mwc1t1_reoriented_fsl.nii.gz
folderbase: 128_S_0216_ADNI-T1_2006-09-12_07_31_03.0
ott_students/ephemeral/sankeith/scott_10k_20aug25/128_S_0216_ADNI-T1_2006-09-12_07_31_03.0
 -i /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_20aug25/128_S_0216_ADNI-T1_2006-09-12_07_31_03.0/Z_F_1mm_128_S_0216_ADNI-T1_2006-09-12_07_31_03.0_dicv_mwc1t1_reoriented_fsl.nii.gz --label=/rds/general/project/c3nl_scott_students/live/sankeith/standards/MNI152_1mm_destrieux_aparc.a2009+aseg.nii.gz -o /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_20aug25/128_S_0216_ADNI-T1_2006-09-12_07_31_03.0/destrieux_mwc1t1_128_S_0216_ADNI-T1_2006-09-12_07_31_03.0.txt
fslmeants -i /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_20aug25/128_S_0216_ADNI-T1_2006-09-12_07_31_03.0/Z_F_1m

### Rerun parcellation code to get raw parcellation stats

In [11]:
%%bash

outdir=/rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_20aug25/
destrieux_file=/rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_lists/successful_destrieux_parcellations.txt
desikan_file=/rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_lists/successful_desikan_parcellations.txt

>"$destrieux_file"
>"$desikan_file"

cd "$outdir"
find "$outdir" -type f -name "destrieux_mwc1t1*" >> "$destrieux_file"
find "$outdir" -type f -name "desikan_mwc1t1*" >> "$desikan_file"

wc -l< "$destrieux_file"
wc -l< "$desikan_file"

15725
15725


In [2]:
output_file='/rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_lists/successful_zscored_mwc1t1s.txt'

zscored_codes = {}
with open(output_file) as zscored_file:
    file = zscored_file.readlines()
    for path in file:
        patient_name_parts = path.split('/')
        patient_name = patient_name_parts[-2]
        patient_name = patient_name.replace('\n','')
        patient_name = patient_name.strip()
        zscored_codes.update({patient_name:path})

#### Rerun Desikan

In [3]:
output_file='/rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_lists/successful_desikan_parcellations.txt'

desikan_successes = []
with open(output_file) as desikan_file:
    file = desikan_file.readlines()
    for path in file:
        patient_name_parts = path.split('/')
        patient_name = patient_name_parts[-2]
        patient_name = patient_name.replace('\n','')
        patient_name = patient_name.strip()
        desikan_successes.append(patient_name)
print(desikan_successes[:5])

['128_S_0216_ADNI-T1_2006-09-12_07_31_03.0', '037_S_5162_ADNI-T1_2013-05-03_08_21_20.0', '128_S_0216_ADNI-T1_2007-02-22_10_49_53.0', '128_S_0216_ADNI-T1_2008-03-12_14_25_20.0', '033_S_0513_ADNI-T1_2006-05-18_11_34_42.0']


In [4]:
import os
missing_desikan_list = '/rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_lists/missing_desikan_list.txt'
counter = 0

!>"$missing_desikan_list"
path_list = []
for zscored in zscored_codes.keys():
    if zscored not in desikan_successes:
        path = zscored_codes[zscored]
        path_list.append(path)
        with open(missing_desikan_list,'a') as outfile:
            path.replace('\n','')
            if os.path.exists(path.replace('\n','')):
                outfile.write(path)

!wc -l< "$missing_desikan_list"

5726


In [5]:
%%file /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/rerun_desikan.sh

#!/bin/bash
#PBS -l walltime=00:10:00
#PBS -l select=1:ncpus=1:mem=20gb
#PBS -N rerun_desikan
#PBS -J 0-5810
#PBS -o /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_logs/b2c/
#PBS -e /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_logs/b2c/

module load tools/prod
module load fsl > /dev/null 2>&1
module load FSL/6.0.7.17 > /dev/null 2>&1
source ${FSLDIR}/etc/fslconf/fsl.sh
export FSLOUTPUTTYPE=NIFTI_GZ

inpath=`head -n ${PBS_ARRAY_INDEX} /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_lists/missing_desikan_list.txt | tail -n 1`
dk_atlas=/rds/general/project/c3nl_scott_students/live/sankeith/standards/MNI152_1mm_desikan_aparc+aseg.nii.gz
outdir=/rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_20aug25
folderbase=$(basename "$(dirname "$inpath")")
outroot="$outdir"/"$folderbase"
echo "inpath: $inpath";
echo "folderbase: $folderbase";
echo "outroot: $outroot"

cmd="fslmeants -i $inpath --label=$dk_atlas -o ${outroot}/desikan_mwc1t1_${folderbase}.txt"
echo ${cmd}
eval ${cmd}

Writing /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/rerun_desikan.sh


In [6]:
%%bash

chmod -Rf 775 /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/rerun_desikan.sh
export PBS_ARRAY_INDEX=1; /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/rerun_desikan.sh
qsub /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/rerun_desikan.sh

inpath: /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_20aug25/018_S_0142_ADNI-T1_2008-03-03_10_51_19.0/Z_F_1mm_018_S_0142_ADNI-T1_2008-03-03_10_51_19.0_dicv_mwc1t1_reoriented_fsl.nii.gz
folderbase: 018_S_0142_ADNI-T1_2008-03-03_10_51_19.0
ott_students/ephemeral/sankeith/scott_10k_20aug25/018_S_0142_ADNI-T1_2008-03-03_10_51_19.0
 -i /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_20aug25/018_S_0142_ADNI-T1_2008-03-03_10_51_19.0/Z_F_1mm_018_S_0142_ADNI-T1_2008-03-03_10_51_19.0_dicv_mwc1t1_reoriented_fsl.nii.gz --label=/rds/general/project/c3nl_scott_students/live/sankeith/standards/MNI152_1mm_desikan_aparc+aseg.nii.gz -o /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_20aug25/018_S_0142_ADNI-T1_2008-03-03_10_51_19.0/desikan_mwc1t1_018_S_0142_ADNI-T1_2008-03-03_10_51_19.0.txt
2122678[].pbs-7


#### Rerun Destrieux

In [7]:
output_file='/rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_lists/successful_destrieux_parcellations.txt'

destrieux_successes = []
with open(output_file) as destrieux_file:
    file = destrieux_file.readlines()
    for path in file:
        patient_name_parts = path.split('/')
        patient_name = patient_name_parts[-2]
        patient_name = patient_name.replace('\n','')
        patient_name = patient_name.strip()
        destrieux_successes.append(patient_name)
print(destrieux_successes[:5])

['128_S_0216_ADNI-T1_2006-09-12_07_31_03.0', '037_S_5162_ADNI-T1_2013-05-03_08_21_20.0', '128_S_0216_ADNI-T1_2007-02-22_10_49_53.0', '128_S_0216_ADNI-T1_2008-03-12_14_25_20.0', '033_S_0513_ADNI-T1_2006-05-18_11_34_42.0']


In [8]:
import os
missing_destrieux_list = '/rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_lists/missing_destrieux_list.txt'
counter = 0

!>"$missing_destrieux_list"
path_list = []
for zscored in zscored_codes.keys():
    if zscored not in destrieux_successes:
        path = zscored_codes[zscored]
        path_list.append(path)
        with open(missing_destrieux_list,'a') as outfile:
            path.replace('\n','')
            if os.path.exists(path.replace('\n','')):
                outfile.write(path)

!wc -l< "$missing_destrieux_list"

5726


In [9]:
%%file /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/rerun_destrieux.sh

#!/bin/bash
#PBS -l walltime=00:10:00
#PBS -l select=1:ncpus=1:mem=20gb
#PBS -N rerun_destrieux
#PBS -J 0-5800
#PBS -o /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_logs/b2c/
#PBS -e /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_logs/b2c/

module load tools/prod
module load fsl > /dev/null 2>&1
module load FSL/6.0.7.17 > /dev/null 2>&1
source ${FSLDIR}/etc/fslconf/fsl.sh
export FSLOUTPUTTYPE=NIFTI_GZ

inpath=`head -n ${PBS_ARRAY_INDEX} /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_lists/missing_destrieux_list.txt | tail -n 1`
des_atlas=/rds/general/project/c3nl_scott_students/live/sankeith/standards/MNI152_1mm_destrieux_aparc.a2009+aseg.nii.gz
outdir=/rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_20aug25
folderbase=$(basename "$(dirname "$inpath")")
outroot="$outdir"/"$folderbase"
echo "inpath: $inpath";
echo "folderbase: $folderbase";
echo "outroot: $outroot"

cmd="fslmeants -i $inpath --label=$des_atlas -o ${outroot}/destrieux_mwc1t1_${folderbase}.txt"
echo ${cmd}
eval ${cmd}

Writing /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/rerun_destrieux.sh


In [10]:
%%bash

chmod -Rf 775 /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/rerun_destrieux.sh
export PBS_ARRAY_INDEX=1; /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/rerun_destrieux.sh
qsub /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/rerun_destrieux.sh

inpath: /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_20aug25/018_S_0142_ADNI-T1_2008-03-03_10_51_19.0/Z_F_1mm_018_S_0142_ADNI-T1_2008-03-03_10_51_19.0_dicv_mwc1t1_reoriented_fsl.nii.gz
folderbase: 018_S_0142_ADNI-T1_2008-03-03_10_51_19.0
ott_students/ephemeral/sankeith/scott_10k_20aug25/018_S_0142_ADNI-T1_2008-03-03_10_51_19.0
 -i /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_20aug25/018_S_0142_ADNI-T1_2008-03-03_10_51_19.0/Z_F_1mm_018_S_0142_ADNI-T1_2008-03-03_10_51_19.0_dicv_mwc1t1_reoriented_fsl.nii.gz --label=/rds/general/project/c3nl_scott_students/live/sankeith/standards/MNI152_1mm_destrieux_aparc.a2009+aseg.nii.gz -o /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_20aug25/018_S_0142_ADNI-T1_2008-03-03_10_51_19.0/destrieux_mwc1t1_018_S_0142_ADNI-T1_2008-03-03_10_51_19.0.txt
2122686[].pbs-7


### Scrape Desikan stats

#### Split successful_desikan_parcellations.txt into 10 smaller txts so the scraping can be parallelised

In [12]:
%%bash
rootdir=/rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_lists
cd "$rootdir"
desikan_file=${rootdir}/successful_desikan_parcellations.txt
split -l 1600 -d "$desikan_file"


declare -A split_files
split_files["x00"]="part1_successful_desikan_parcellations"
split_files["x01"]="part2_successful_desikan_parcellations"
split_files["x02"]="part3_successful_desikan_parcellations"
split_files["x03"]="part4_successful_desikan_parcellations"
split_files["x04"]="part5_successful_desikan_parcellations"
split_files["x05"]="part6_successful_desikan_parcellations"
split_files["x06"]="part7_successful_desikan_parcellations"
split_files["x07"]="part8_successful_desikan_parcellations"
split_files["x08"]="part9_successful_desikan_parcellations"
split_files["x09"]="part10_successful_desikan_parcellations"

for file in "${!split_files[@]}"; do
    start_path=${rootdir}/${file}
    renamed_path=${rootdir}/${split_files[${file}]}.txt
    mv "$start_path" "$renamed_path"
done

In [13]:
%%file /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/parallel_scrape_desikan_stats.sh

#!/bin/bash
#PBS -l walltime=12:00:00
#PBS -l select=1:ncpus=1:mem=50gb
#PBS -N parallel_scrape_desikan_stats
#PBS -J 1-10
#PBS -o /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_logs/parcellate/
#PBS -e /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_logs/parcellate/

mkdir -p /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_logs/parcellate
module load tools/prod
source /rds/general/user/sk4724/home/venv_1/bin/activate

python /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/parallel_scrape_desikan_stats.py ${PBS_ARRAY_INDEX}

Writing /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/parallel_scrape_desikan_stats.sh


In [14]:
%%file /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/parallel_scrape_desikan_stats.py

import os
import pandas as pd
import sys
import tqdm
from tqdm import tqdm

file_index = int(sys.argv[1]) ## PBS ARRAY INDEX is passed with this
print(file_index)

desikan_file= f'/rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_lists/part{file_index}_successful_desikan_parcellations.txt'

desikan_dfs = []
with open(desikan_file, 'r') as desikan_file:
    lines = desikan_file.readlines()
    for line in tqdm(lines, desc = f'Scraping from part {file_index} of the {len(lines)} successful desikan parcellation filepaths...: '):
        data_key = line.split('/')[-2]
        path = line.strip('\n')
        df = pd.read_csv(path, sep=r'\s+', header=None)
        df['DATA_KEY'] = data_key
        desikan_dfs.append(df)
        combined_df = pd.concat(desikan_dfs, axis=0)

cols = combined_df.columns
combined_df = combined_df[[cols[-1]] + list(cols[:-1])]
print(f"Combined shape: {combined_df.shape}")
combined_df.to_csv(f'/rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_lists/part{file_index}_scraped_desikan_stats.csv',index=False)

Writing /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/parallel_scrape_desikan_stats.py


In [15]:
%%bash

chmod -Rf 775 /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/parallel_scrape_desikan_stats.*
#export PBS_ARRAY_INDEX=1; /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/parallel_scrape_desikan_stats.sh
qsub /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/parallel_scrape_desikan_stats.sh

2125397[].pbs-7


#### Scrape Destrieux stats

In [16]:
%%bash
rootdir=/rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_lists
cd "$rootdir"
destrieux_file=${rootdir}/successful_destrieux_parcellations.txt
split -l 1600 -d "$destrieux_file"


declare -A split_files
split_files["x00"]="part1_successful_destrieux_parcellations"
split_files["x01"]="part2_successful_destrieux_parcellations"
split_files["x02"]="part3_successful_destrieux_parcellations"
split_files["x03"]="part4_successful_destrieux_parcellations"
split_files["x04"]="part5_successful_destrieux_parcellations"
split_files["x05"]="part6_successful_destrieux_parcellations"
split_files["x06"]="part7_successful_destrieux_parcellations"
split_files["x07"]="part8_successful_destrieux_parcellations"
split_files["x08"]="part9_successful_destrieux_parcellations"
split_files["x09"]="part10_successful_destrieux_parcellations"

for file in "${!split_files[@]}"; do
    start_path=${rootdir}/${file}
    renamed_path=${rootdir}/${split_files[${file}]}.txt
    mv "$start_path" "$renamed_path"
done

In [17]:
%%file /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/parallel_scrape_destrieux_stats.sh

#!/bin/bash
#PBS -l walltime=72:00:00
#PBS -l select=1:ncpus=1:mem=50gb
#PBS -N parallel_scrape_destrieux_stats
#PBS -J 1-10
#PBS -o /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_logs/parcellate/
#PBS -e /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_logs/parcellate/

mkdir -p /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_logs/parcellate
module load tools/prod
source /rds/general/user/sk4724/home/venv_1/bin/activate

python /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/parallel_scrape_destrieux_stats.py ${PBS_ARRAY_INDEX}

Writing /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/parallel_scrape_destrieux_stats.sh


In [18]:
%%file /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/parallel_scrape_destrieux_stats.py

import os
import pandas as pd
import sys
import tqdm
from tqdm import tqdm

file_index = int(sys.argv[1]) ## PBS ARRAY INDEX is passed with this
print(file_index)

destrieux_file= f'/rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_lists/part{file_index}_successful_destrieux_parcellations.txt'

destrieux_dfs = []
with open(destrieux_file, 'r') as destrieux_file:
    lines = destrieux_file.readlines()
    for line in tqdm(lines, desc = f'Scraping from part {file_index} of the {len(lines)} successful destrieux parcellation filepaths...: '):
        data_key = line.split('/')[-2]
        path = line.strip('\n')
        df = pd.read_csv(path, sep=r'\s+', header=None)
        df['DATA_KEY'] = data_key
        destrieux_dfs.append(df)
        combined_df = pd.concat(destrieux_dfs, axis=0)

cols = combined_df.columns
combined_df = combined_df[[cols[-1]] + list(cols[:-1])]
print(f"Combined shape: {combined_df.shape}")
combined_df.to_csv(f'/rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_lists/part{file_index}_scraped_destrieux_stats.csv',index=False)

Writing /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/parallel_scrape_destrieux_stats.py


In [19]:
%%bash

chmod -Rf 775 /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/parallel_scrape_destrieux_stats.*
#export PBS_ARRAY_INDEX=1; /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/parallel_scrape_destrieux_stats.sh
qsub /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/parallel_scrape_destrieux_stats.sh

2125399[].pbs-7


#### Concatenate the small csv files we have just made to create the full sets of patientwise ROIs we actually need

In [1]:
%%file /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/concat_scraped_stats.py

import os
import pandas as pd

root_dir = '/rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_lists/'
desikan_dfs = []
destrieux_dfs = []
for i in range(1,11):
    desikan_part = os.path.join(root_dir,f"part{i}_scraped_desikan_stats.csv")
    desikan_part_df = pd.read_csv(desikan_part)
    desikan_dfs.append(desikan_part_df)
    
    destrieux_part = os.path.join(root_dir,f"part{i}_scraped_destrieux_stats.csv")
    destrieux_part_df = pd.read_csv(destrieux_part)
    destrieux_dfs.append(destrieux_part_df)

desikan_df = pd.concat(desikan_dfs)
print(f"Desikan dataframe is of shape {desikan_df.shape}")
destrieux_df = pd.concat(destrieux_dfs)
print(f"Destrieux dataframe is of shape {destrieux_df.shape}")

desikan_df.to_csv("/rds/general/project/c3nl_scott_students/live/sankeith/scott_10k_housekeeping/scraped_desikan_stats.csv", index = False)
destrieux_df.to_csv("/rds/general/project/c3nl_scott_students/live/sankeith/scott_10k_housekeeping/scraped_destrieux_stats.csv", index = False)

Writing /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/concat_scraped_stats.py


In [3]:
%%file /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/concat_scraped_stats.sh

#!/bin/bash
#PBS -l walltime=12:00:00
#PBS -l select=1:ncpus=1:mem=30gb
#PBS -N concat_scraped_stats
#PBS -o /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_logs/parcellate/
#PBS -e /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_logs/parcellate/

mkdir -p /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_logs/parcellate
module load tools/prod
source /rds/general/user/sk4724/home/venv_1/bin/activate

python /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/concat_scraped_stats.py

Writing /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/concat_scraped_stats.sh


In [4]:
%%bash

chmod -Rf 775 /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/concat_scraped_stats.*
qsub /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/concat_scraped_stats.sh

2575598.pbs-7


In [5]:
%%file /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/clean_scraped_stats.py

import pandas as pd
import os, pickle

##Running this code on the HPC or else it crashes the kernel

root = "/rds/general/project/c3nl_scott_students/live/sankeith/scott_10k_housekeeping/"
lut_dict = pickle.load(open("/rds/general/project/c3nl_scott_students/live/sankeith/standards/lut_dict.pkl", 'rb'))
atlases = ["desikan", "destrieux"]
non_gm_cols = ['Left-Cerebral-White-Matter', ##NOTE TO SELF - THIS SHOULD ONLY CONTAIN THE COLUMNS YOU WANT DROPPED
               'Left-Lateral-Ventricle',
               'Left-Inf-Lat-Vent',
               'Left-Cerebellum-White-Matter',
               '3rd-Ventricle',
               '4th-Ventricle',
               'Brain-Stem',
               'CSF',
               'Left-vessel',
               'Left-choroid-plexus',
               'Right-Cerebral-White-Matter',
               'Right-Lateral-Ventricle',
               'Right-Inf-Lat-Vent',
               'Right-Cerebellum-White-Matter',
               'Right-vessel',
               'Right-choroid-plexus',
               'WM-hypointensities',
               'Optic-Chiasm',
               'CC_Posterior',
               'CC_Mid_Posterior',
               'CC_Central',
               'CC_Mid_Anterior',
               'CC_Anterior']

###QC check so I can confirm non_gm_cols has len = 23
assert len(non_gm_cols) == 23

##Pickle non_gm_cols for use later
pickle.dump(non_gm_cols, open("/rds/general/project/c3nl_scott_students/live/sankeith/standards/non_gm_cols.pkl", 'wb'))

for atlas in atlases:
    stats_path = os.path.join(root, f"scraped_{atlas}_stats.csv")
    df = pd.read_csv(stats_path, low_memory = False)

    ##Rename columns with the FreeSurfer LIT dictionary I pickled earlier
    pickle_path = f"/rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_parcellate/atlas_roi_extraction/{atlas}+aseg_extraction.pkl"
    pickled_column_names = pickle.load(open(pickle_path, 'rb'))
    pickled_int_column_names = [int(x) for x in pickled_column_names]
    new_column_names = ['DATA_KEY'] + pickled_int_column_names
    df.columns = new_column_names
    assert df.columns.tolist() == new_column_names
    df.rename(columns = lut_dict, inplace = True)
    df.to_csv(os.path.join(root, f"renamed_cols_scraped_{atlas}_stats.csv"), index = False)

    ##After renaming, drop any 'Unknown columns'
    df_dropped = df.drop(columns = 'Unknown')
    print(df_dropped.shape)
    df_dropped.to_csv(os.path.join(root, f"dropped_unknown_cols_scraped_{atlas}_stats.csv"), index = False)

    ##Further cleaning to keep only grey matter ROIs
    gm_only_df = df_dropped.drop(axis = 1, labels = non_gm_cols)
    print(gm_only_df.shape)
    gm_only_df.to_csv(os.path.join(root, f"gm_only_scraped_{atlas}_stats.csv"), index = False)

Writing /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/clean_scraped_stats.py


In [7]:
%%file /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/clean_scraped_stats.sh

#!/bin/bash
#PBS -l walltime=00:30:00
#PBS -l select=1:ncpus=1:mem=30gb
#PBS -N clean_scraped_stats
#PBS -o /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_logs/parcellate/
#PBS -e /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_logs/parcellate/

mkdir -p /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_logs/parcellate
module load tools/prod
source /rds/general/user/sk4724/home/venv_1/bin/activate

python /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/clean_scraped_stats.py

Writing /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/clean_scraped_stats.sh


In [9]:
%%bash

chmod -Rf 775 /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/clean_scraped_stats.*
qsub /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_scripts/parcellate/clean_scraped_stats.sh

2576649.pbs-7


#### ROI extraction from neuromaps

1. Take neuromaps
2. Run them through fslmeants with both atlases
3. Have transposed and non-transposed versions
4. Label rows and columns with their ROI name and what neuromap they came from

In [2]:
%%bash

des_atlas=/rds/general/project/c3nl_scott_students/live/sankeith/standards/MNI152_1mm_destrieux_aparc.a2009+aseg.nii.gz
dk_atlas=/rds/general/project/c3nl_scott_students/live/sankeith/standards/MNI152_1mm_desikan_aparc+aseg.nii.gz
neuromaps=/rds/general/project/c3nl_scott_students/live/sankeith/regression_generator/2_FINAL_NEUROMAPS_REGRESSOR.nii.gz
flipped_neuromaps=/rds/general/project/c3nl_scott_students/live/sankeith/regression_generator/flipped_2_FINAL_NEUROMAPS_REGRESSOR.nii.gz
outpath=/rds/general/project/c3nl_scott_students/live/sankeith/regression_generator

module purge
module load tools/prod
module load FSL/6.0.7.17 > /dev/null 2>&1
module load FreeSurfer/7.4.1-centos8_x86_64 > /dev/null 2>&1

source $FSLDIR/etc/fslconf/fsl.sh
export FSLOUTPUTTYPE=NIFTI_GZ

cmd="fslmeants -i $neuromaps --label=$des_atlas -o ${outpath}/destrieux_neuromaps.txt"
echo ${cmd};
eval ${cmd}

cmd2="fslmeants -i $neuromaps --label=$dk_atlas -o ${outpath}/desikan_neuromaps.txt"
echo ${cmd2};
eval ${cmd2}

cmd3="fslmeants -i $flipped_neuromaps --label=$des_atlas -o ${outpath}/flipped_destrieux_neuromaps.txt"
echo ${cmd3};
eval ${cmd3}

cmd4="fslmeants -i $flipped_neuromaps --label=$dk_atlas -o ${outpath}/flipped_desikan_neuromaps.txt"
echo ${cmd4};
eval ${cmd4}

fslmeants -i /rds/general/project/c3nl_scott_students/live/sankeith/regression_generator/2_FINAL_NEUROMAPS_REGRESSOR.nii.gz --label=/rds/general/project/c3nl_scott_students/live/sankeith/standards/MNI152_1mm_destrieux_aparc.a2009+aseg.nii.gz -o /rds/general/project/c3nl_scott_students/live/sankeith/regression_generator/destrieux_neuromaps.txt
fslmeants -i /rds/general/project/c3nl_scott_students/live/sankeith/regression_generator/2_FINAL_NEUROMAPS_REGRESSOR.nii.gz --label=/rds/general/project/c3nl_scott_students/live/sankeith/standards/MNI152_1mm_desikan_aparc+aseg.nii.gz -o /rds/general/project/c3nl_scott_students/live/sankeith/regression_generator/desikan_neuromaps.txt
fslmeants -i /rds/general/project/c3nl_scott_students/live/sankeith/regression_generator/flipped_2_FINAL_NEUROMAPS_REGRESSOR.nii.gz --label=/rds/general/project/c3nl_scott_students/live/sankeith/standards/MNI152_1mm_destrieux_aparc.a2009+aseg.nii.gz -o /rds/general/project/c3nl_scott_students/live/sankeith/regression_g

In [3]:
import os, pickle
import pandas as pd
import numpy as np

neuromaps_dict = {
    0:"D1",
    1:"D2",
    2:"DAT",
    3:"NET",
    4:"5HT1A",
    5:"5HT1B",
    6:"5HT2A",
    7:"5HT4",
    8:"5HT6",
    9:"5HTT",
    10:"a4b2",
    11:"M1",
    12:"vAChT",
    13:"NMDA",
    14:"mGluR5",
    15:"GABAA/BZ",
    16:"H3",
    17:"CB1",
    18:"MOR"
}

root = "/rds/general/project/c3nl_scott_students/live/sankeith/regression_generator/"
txts = ["desikan_neuromaps", "destrieux_neuromaps", "flipped_desikan_neuromaps", "flipped_destrieux_neuromaps"]
lut_dict = pickle.load(open('/rds/general/project/c3nl_scott_students/live/sankeith/standards/lut_dict.pkl', 'rb'))
non_gm_cols = pickle.load(open('/rds/general/project/c3nl_scott_students/live/sankeith/standards/non_gm_cols.pkl', 'rb'))

for txt in txts:
    txtpath = os.path.join(root + txt + '.txt')
    pickle_key = txt.split('_')[0]
    print(pickle_key)
    pickle_path = f"/rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_parcellate/atlas_roi_extraction/{pickle_key}+aseg_extraction.pkl"
        
    ## New_column_names are the list of numbers from running fslmeants -in atlas --label=atlas -o atlas
    try:
        new_column_names = pickle.load(open(pickle_path, 'rb'))
    except FileNotFoundError:
        print("using flipped neuromaps values...")
        pickle_key = txt.split('_')[1]
        print(pickle_key)
        pickle_path = f"/rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_parcellate/atlas_roi_extraction/{pickle_key}+aseg_extraction.pkl"
        new_column_names = pickle.load(open(pickle_path, 'rb'))

    for col in new_column_names:
        new_column_names = [int(x) for x in new_column_names]

    ## Turn neuromaps txt into csv
    if os.path.exists:
        print(f"\nPath to the txt {txt} exists. Continuing...")
    df = pd.read_csv(txtpath, header = None, sep = r'\s+')

    ##Rename neuromaps csv columns with new_column_names and appropriate row names
    print(f"New columns names: {new_column_names[:10]}")
    df.columns = new_column_names
    assert df.columns.tolist() == new_column_names
    print(f"Dataframe column names: {df.columns.tolist()[:10]}")
    df.rename(columns = lut_dict, index = neuromaps_dict, inplace = True)
    print(f"Final Dataframe column names: {df.columns.tolist()[:10]}")
    print(df.head(5))

    ## Save renamed neuromaps df to csv
    df.to_csv(os.path.join(root + txt + '.csv'))
    
    ## Make a transposed version if we need it
    df_transpose = df.transpose()
    print(df_transpose.head(2))
    df_transpose.to_csv(os.path.join(root + 'transposed_' + txt + '.csv'))

    ## Make a non-transposed and transposed version where 'Unknown' columns are cleaned out
    df_dropped = df.drop("Unknown", axis = 'columns')
    print(df_dropped.shape)
    df_dropped.to_csv(os.path.join(root + 'dropped_unknown_cols_' + txt + '.csv'))
    df_dropped_transpose = df_dropped.transpose()
    print(df_dropped_transpose.head(2))
    df_dropped_transpose.to_csv(os.path.join(root + 'dropped_unknown_cols_transposed_' + txt + '.csv'))

    ##Further cleaning to keep only grey matter ROIs
    gm_only_df = df_dropped.drop(axis = 1, labels = non_gm_cols)
    print(gm_only_df.shape)
    gm_only_df.to_csv(os.path.join(root + 'gm_only_' + txt + '.csv'))
    gm_only_df_transpose = gm_only_df.transpose()
    print(gm_only_df_transpose.head(2))
    gm_only_df_transpose.to_csv(os.path.join(root + 'gm_only_transposed_' + txt + '.csv'))
    print(os.path.join(root + 'gm_only_transposed_' + txt + '.csv'))



desikan

Path to the txt desikan_neuromaps exists. Continuing...
New columns names: [0, 2, 0, 4, 5, 0, 7, 8, 0, 10]
Dataframe column names: [0, 2, 0, 4, 5, 0, 7, 8, 0, 10]
Final Dataframe column names: ['Unknown', 'Left-Cerebral-White-Matter', 'Unknown', 'Left-Lateral-Ventricle', 'Left-Inf-Lat-Vent', 'Unknown', 'Left-Cerebellum-White-Matter', 'Left-Cerebellum-Cortex', 'Unknown', 'Left-Thalamus-Proper']
       Unknown  Left-Cerebral-White-Matter  Unknown  Left-Lateral-Ventricle  \
D1           0                   -0.048509        0               -0.259580   
D2           0                   -0.047406        0                0.358097   
DAT          0                    0.320754        0               -0.477203   
NET          0                    0.007859        0               -1.360071   
5HT1A        0                   -0.053717        0               -0.797046   

       Left-Inf-Lat-Vent  Unknown  Left-Cerebellum-White-Matter  \
D1              0.161737        0                   

In [13]:
df

,Unknown,Left-Cerebral-White-Matter,Unknown,Left-Lateral-Ventricle,Left-Inf-Lat-Vent,Unknown,Left-Cerebellum-White-Matter,Left-Cerebellum-Cortex,Unknown,Left-Thalamus-Proper,...,ctx_rh_S_parieto_occipital,ctx_rh_S_pericallosal,ctx_rh_S_postcentral,ctx_rh_S_precentral-inf-part,ctx_rh_S_precentral-sup-part,ctx_rh_S_suborbital,ctx_rh_S_subparietal,ctx_rh_S_temporal_inf,ctx_rh_S_temporal_sup,ctx_rh_S_temporal_transverse
D1,0,-0.180151,0,-0.208560,-0.141550,0,-0.425207,-0.019691,0,-0.635307,...,0.384161,-0.483454,0.183012,-0.460020,0.400084,0.092383,-0.222477,-0.482401,-0.148968,0.613915
D2,0,0.341669,0,0.406613,-0.254200,0,-0.315326,-0.447974,0,-0.086951,...,0.545999,1.834791,-0.425516,-0.020489,-0.296482,-0.337145,1.336809,0.175742,0.170830,0.786041
DAT,0,0.202253,0,0.417568,0.679578,0,-0.467595,-0.492243,0,0.519202,...,-0.352299,-1.077233,-0.147745,0.130932,-0.138410,0.343767,-0.313221,-0.022923,-0.082015,-0.027916
NET,0,-0.013381,0,-1.420133,-0.671359,0,-0.408926,-0.104747,0,1.766540,...,0.322885,-0.861266,0.944793,0.158785,0.362940,0.006534,0.513023,-0.184360,0.215379,1.157443
5HT1A,0,-0.265943,0,0.093057,-0.540938,0,-0.486682,-0.271573,0,-0.387436,...,-0.521790,-0.040788,0.417224,-0.740569,0.825916,-0.460433,0.105946,-0.625581,-0.400169,-0.666983
5HT1B,0,-0.048939,0,-0.323021,-0.411324,0,-0.508924,-0.151509,0,-0.720560,...,-0.328358,-0.448671,-0.330516,1.453054,-0.188460,-0.673395,-0.328466,-0.521391,-0.404575,-0.764042
5HT2A,0,-0.316310,0,-0.424748,0.030922,0,-0.767554,-0.700978,0,-0.206028,...,-0.148553,0.193303,-0.266438,0.789420,0.594459,0.590770,0.646633,0.825962,0.358424,1.237392
5HT4,0,-0.279037,0,-0.180029,0.212974,0,-0.678935,-0.618376,0,0.490663,...,-0.243469,0.025607,-0.269255,0.349784,0.142305,0.298646,0.365387,0.594913,0.204749,0.728094
5HT6,0,-0.264652,0,-0.183390,-0.165343,0,0.863036,1.274014,0,-0.702171,...,-0.138812,-0.159818,-0.513413,0.815284,-0.326982,-1.127850,-0.145673,-0.388991,-0.329847,-0.421801
5HTT,0,-0.267545,0,-0.177955,0.220198,0,-0.454020,-0.472111,0,3.721393,...,-0.268829,0.218409,-0.311397,0.035989,-0.086874,0.140394,0.074606,-0.080241,-0.111189,0.613099
